# <font color="#418FDE" size="6.5" uppercase>**Meta Ensembles**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply AdaBoost and voting ensembles to combine base estimators. 
- Build stacked classifiers and regressors using cross-validated meta-features. 
- Prevent leakage when designing ensemble workflows. 


## **1. AdaBoost Voting**

### **1.1. AdaBoost Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_B/image_01_01.jpg?v=1784012390" width="250">



>* Combines weak learners into stronger models
>* Focuses later learners on earlier mistakes

>* AdaBoost shifts focus toward harder examples
>* Errors guide later learners to improve

>* Stronger learners get larger final votes
>* Sequential weak rules form one accurate ensemble



In [ ]:
#@title Python Code - AdaBoost Basics

# This example trains a small AdaBoost classifier.
# AdaBoost combines weak learners with adaptive weighting.
# The output compares one tree with boosted trees.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate that features and labels match.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Split data before training to avoid test leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Train one weak learner for a simple baseline.
weak_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
weak_tree.fit(X_train, y_train)

# Train AdaBoost using many shallow decision trees.
adaboost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=50,
    learning_rate=0.8,
    random_state=42,
)

adaboost_model.fit(X_train, y_train)
weak_accuracy = accuracy_score(y_test, weak_tree.predict(X_test))
boost_accuracy = accuracy_score(y_test, adaboost_model.predict(X_test))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Single weak tree test accuracy: {weak_accuracy:.3f}")
print(f"AdaBoost ensemble test accuracy: {boost_accuracy:.3f}")
print("AdaBoost improves by reweighting mistakes across weak learners.")



### **1.2. Hard Voting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_B/image_01_02.jpg?v=1784012392" width="250">



>* Combines classifier labels by majority vote
>* Uses model diversity to reduce errors

>* Uses labels, not probability estimates
>* Ignores confidence unless models are weighted

>* Use accurate, diverse models for stronger voting
>* Test separately to avoid overfit confidence



In [ ]:
#@title Python Code - Hard Voting

# Demonstrate hard voting with three simple classifiers.
# Each model votes for one class label.
# The majority vote becomes the ensemble prediction.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Load a small packaged classification dataset.
data = load_breast_cancer()
features = data.data
target = data.target

# Check that features and labels match.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split before scaling to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42
)

# Build diverse classifiers for majority voting.
logistic_model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
knn_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))

# Combine final class labels using hard voting.
hard_voter = VotingClassifier(
    estimators=[("logistic", logistic_model), ("tree", tree_model), ("knn", knn_model)],
    voting="hard",
)

# Train the ensemble and compare it with members.
hard_voter.fit(X_train, y_train)
member_scores = []
for name, model in hard_voter.named_estimators_.items():
    score = accuracy_score(y_test, model.predict(X_test))
    member_scores.append((name, round(score, 3)))

# Show one test case where votes are counted.
sample = X_test[:1]
votes = []
for name, model in hard_voter.named_estimators_.items():
    label = data.target_names[model.predict(sample)[0]]
    votes.append(name + ": " + label)

ensemble_label = data.target_names[hard_voter.predict(sample)[0]]
ensemble_score = accuracy_score(y_test, hard_voter.predict(X_test))

print("scikit-learn version:", sklearn.__version__)
print("Member accuracies:", member_scores)
print("Votes for one patient:", "; ".join(votes))
print("Hard voting prediction:", ensemble_label)
print("Hard voting test accuracy:", round(ensemble_score, 3))



### **1.3. Soft Voting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_B/image_01_03.jpg?v=1784012394" width="250">



>* Combines models using predicted class probabilities
>* Uses confidence for more nuanced decisions

>* Reliable probabilities make soft voting effective
>* Overconfident models can distort ensemble decisions

>* Combine diverse model probabilities for steadier predictions
>* Choose voting weights using validation data



In [ ]:
#@title Python Code - Soft Voting

# This example demonstrates soft voting with probabilities.
# Three classifiers combine confidence scores for predictions.
# The output compares individual and ensemble accuracy.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Load a small packaged binary classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate that features and labels match correctly.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Split before fitting to avoid test data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Build simple probability-producing base classifiers.
logistic_model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
knn_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)

# Soft voting averages predicted class probabilities.
soft_voter = VotingClassifier(
    estimators=[
        ("logistic", logistic_model),
        ("knn", knn_model),
        ("tree", tree_model),
    ],
    voting="soft",
)

# Fit each model only on the training data.
models = [
    ("Logistic regression", logistic_model),
    ("K-nearest neighbors", knn_model),
    ("Decision tree", tree_model),
    ("Soft voting ensemble", soft_voter),
]

print("scikit-learn version:", sklearn.__version__)

# Compare test accuracy for each individual model.
for model_name, model in models:
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(model_name + " accuracy:", round(accuracy, 3))

# Show how the ensemble combines probabilities for one case.
case_index = 0
probabilities = soft_voter.predict_proba(X_test[[case_index]])[0]
print("Soft-voted probabilities:", [round(value, 3) for value in probabilities])
print("Soft-voted class:", data.target_names[soft_voter.predict(X_test[[case_index]])[0]])



## **2. Stacking Models**

### **2.1. Voting Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_B/image_02_01.jpg?v=1784012398" width="250">



>* Combine multiple regressors by averaging predictions
>* Model diversity improves prediction stability

>* Different models act like expert estimators
>* Averaging predictions reduces extremes and variance

>* Voting averages; stacking learns prediction combinations
>* Choose diverse, strong models and sensible weights



In [ ]:
#@title Python Code - Voting Regression

# Demonstrate voting regression with diverse simple models.
# Average predictions can stabilize regression estimates.
# Compare individual errors with the ensemble error.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

# Create a small deterministic regression dataset.
features, target = make_regression(
    n_samples=300,
    n_features=6,
    noise=18.0,
    random_state=42,
)

# Add a gentle nonlinear pattern for model diversity.
target = target + 25.0 * np.sin(features[:, 0])

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# Split data before fitting any model.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Define diverse regressors with simple settings.
linear_model = LinearRegression()
tree_model = DecisionTreeRegressor(max_depth=5, random_state=42)
neighbor_model = KNeighborsRegressor(n_neighbors=7)

# VotingRegressor averages the base model predictions.
voting_model = VotingRegressor(
    estimators=[
        ("linear", linear_model),
        ("tree", tree_model),
        ("neighbors", neighbor_model),
    ]
)

# Fit the ensemble once on the training data.
voting_model.fit(X_train, y_train)

# Evaluate each fitted base model and the voting ensemble.
model_names = ["Linear", "Tree", "Neighbors", "Voting average"]
fitted_models = list(voting_model.estimators_) + [voting_model]
mae_values = []

for fitted_model in fitted_models:
    predictions = fitted_model.predict(X_test)
    mae_values.append(mean_absolute_error(y_test, predictions))

# Show concise results for beginner comparison.
print("scikit-learn version:", sklearn.__version__)
for name, mae in zip(model_names, mae_values):
    print(name + " MAE:", round(mae, 2))

# Show how one prediction is averaged by the ensemble.
sample_predictions = []
for fitted_model in voting_model.estimators_:
    sample_predictions.append(fitted_model.predict(X_test[:1])[0])

print("First test target:", round(y_test[0], 2))
print("Base predictions:", [round(value, 2) for value in sample_predictions])
print("Voting prediction:", round(voting_model.predict(X_test[:1])[0], 2))



### **2.2. Stacked Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_B/image_02_02.jpg?v=1784012396" width="250">



>* Combine diverse classifiers with a meta-classifier
>* Use labels, probabilities, or decision scores

>* Train meta-classifiers on out-of-fold predictions
>* Learn real model strengths without leakage

>* Choose diverse models and simple meta-classifiers
>* Keep preprocessing inside training folds



In [ ]:
#@title Python Code - Stacked Classification

# This script demonstrates stacked classification.
# Cross-validation creates honest meta-features.
# Accuracy compares base and stacked models.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split once to keep final testing separate.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Each base classifier sees the data differently.
logistic_base = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)

tree_base = DecisionTreeClassifier(max_depth=4, random_state=42)
knn_base = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))

# Stacking trains the meta-classifier on out-of-fold predictions.
stacked_model = StackingClassifier(
    estimators=[("logistic", logistic_base), ("tree", tree_base), ("knn", knn_base)],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,
    stack_method="predict_proba",
)

# Fit base models and the stacked model only on training data.
logistic_base.fit(X_train, y_train)
tree_base.fit(X_train, y_train)
knn_base.fit(X_train, y_train)
stacked_model.fit(X_train, y_train)

# Evaluate on untouched test data.
logistic_accuracy = accuracy_score(y_test, logistic_base.predict(X_test))
tree_accuracy = accuracy_score(y_test, tree_base.predict(X_test))
knn_accuracy = accuracy_score(y_test, knn_base.predict(X_test))
stacked_accuracy = accuracy_score(y_test, stacked_model.predict(X_test))

# Print concise results for comparison.
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Base accuracies: logistic={logistic_accuracy:.3f}, tree={tree_accuracy:.3f}, knn={knn_accuracy:.3f}")
print(f"Stacked classifier accuracy: {stacked_accuracy:.3f}")



### **2.3. Stacked Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_B/image_02_03.jpg?v=1784012400" width="250">



>* Combines diverse regressors for continuous predictions
>* Meta-model learns how to blend predictions

>* Train meta-models on out-of-fold predictions
>* Avoid leakage for reliable stacked regression

>* Choose diverse base models and simple blenders
>* Validate workflows carefully to prevent leakage



In [ ]:
#@title Python Code - Stacked Regression

# This example builds a small stacked regressor.
# Cross-validation creates safer meta-features for stacking.
# Test scores compare base models and stacking.

import numpy as np
import sklearn
from sklearn.datasets import make_regression

from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

# A small synthetic regression dataset keeps the lesson reproducible.
features, target = make_regression(
    n_samples=600,
    n_features=8,
    noise=18.0,
    random_state=42,
)

# Add a nonlinear pattern so diverse models can help.
target = target + 35.0 * np.sin(features[:, 0])

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target values.")

# Hold out test data that no model sees during training.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Base regressors make different kinds of prediction errors.
linear_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
tree_model = DecisionTreeRegressor(max_depth=5, random_state=42)
neighbor_model = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=9))

# Stacking trains the final model on cross-validated base predictions.
stacked_model = StackingRegressor(
    estimators=[("ridge", linear_model), ("tree", tree_model), ("knn", neighbor_model)],
    final_estimator=LinearRegression(),
    cv=5,
)

# Fit each model only on the training split.
linear_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)
neighbor_model.fit(X_train, y_train)
stacked_model.fit(X_train, y_train)

# Evaluate on the untouched test split.
linear_mae = mean_absolute_error(y_test, linear_model.predict(X_test))
tree_mae = mean_absolute_error(y_test, tree_model.predict(X_test))
knn_mae = mean_absolute_error(y_test, neighbor_model.predict(X_test))
stacked_mae = mean_absolute_error(y_test, stacked_model.predict(X_test))

# Lower mean absolute error means better continuous predictions.
print("scikit-learn version:", sklearn.__version__)
print("Ridge MAE:", round(linear_mae, 2))
print("Tree MAE:", round(tree_mae, 2))
print("KNN MAE:", round(knn_mae, 2))
print("Stacked regressor MAE:", round(stacked_mae, 2))



## **3. Leakage Safe Stacking**

### **3.1. Cross Validated Stacking**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_B/image_03_01.jpg?v=1784012386" width="250">



>* Train meta-models on out-of-fold predictions
>* Prevent base models from leaking answers

>* Train meta-models on honest predictions
>* Keep training and validation data separate

>* Refit base models after safe meta-training
>* Choose folds matching data structure



In [ ]:
#@title Python Code - Cross Validated Stacking

# This example builds leakage-safe stacking features.
# Out-of-fold predictions train the meta-model honestly.
# Test accuracy compares leaky and safe stacking.

import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.base import clone

# A small synthetic dataset keeps the lesson fast.
features, target = make_classification(
    n_samples=900,
    n_features=12,
    n_informative=6,
    random_state=42,
)

# The test set represents genuinely unseen future data.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.30,
    stratify=target,
    random_state=42,
)

# This check catches accidental shape mistakes early.
if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("Training features and labels must align.")

# One base model and one meta-model keep stacking focused.
base_model = RandomForestClassifier(
    n_estimators=60,
    max_depth=4,
    random_state=42,
)

meta_model = LogisticRegression(max_iter=500, random_state=42)

# Leaky meta-features come from predictions on fitted training rows.
leaky_base = clone(base_model)
leaky_base.fit(X_train, y_train)
leaky_train_meta = leaky_base.predict_proba(X_train)[:, 1].reshape(-1, 1)

leaky_meta = clone(meta_model)
leaky_meta.fit(leaky_train_meta, y_train)
leaky_train_accuracy = accuracy_score(y_train, leaky_meta.predict(leaky_train_meta))

# Safe meta-features are predicted for held-out folds only.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
safe_train_meta = leaky_train_meta.copy()

for train_index, holdout_index in folds.split(X_train, y_train):
    fold_base = clone(base_model)
    fold_base.fit(X_train[train_index], y_train[train_index])
    fold_predictions = fold_base.predict_proba(X_train[holdout_index])[:, 1]
    safe_train_meta[holdout_index, 0] = fold_predictions

safe_meta = clone(meta_model)
safe_meta.fit(safe_train_meta, y_train)

# Final base refitting uses all training rows after safe meta-training.
final_base = clone(base_model)
final_base.fit(X_train, y_train)
test_meta = final_base.predict_proba(X_test)[:, 1].reshape(-1, 1)

safe_test_accuracy = accuracy_score(y_test, safe_meta.predict(test_meta))
leaky_test_accuracy = accuracy_score(y_test, leaky_meta.predict(test_meta))

print("scikit-learn version:", sklearn.__version__)
print("Leaky stacking train accuracy:", round(leaky_train_accuracy, 3))
print("Leaky stacking test accuracy:", round(leaky_test_accuracy, 3))
print("Cross-validated stacking test accuracy:", round(safe_test_accuracy, 3))



### **3.2. Passthrough Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_B/image_03_02.jpg?v=1784012384" width="250">



>* Meta-model uses predictions plus original features
>* More flexibility requires careful leakage control

>* Create meta-features using training folds only
>* Fit preprocessing inside folds to avoid leakage

>* Fit all learned steps within training folds
>* Treat validation folds as unseen future data



In [ ]:
#@title Python Code - Passthrough Features

# This example demonstrates leakage-safe stacking with passthrough.
# Passthrough gives meta-models original features too.
# We compare safe and leaky preprocessing results.

import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Create a small classification dataset with missing values.
features, target = make_classification(
    n_samples=800,
    n_features=8,
    n_informative=5,
    random_state=42,
)

# Add deterministic missing values to require preprocessing.
missing_mask = features[:, 0] > 0.8
features[missing_mask, 1] = float("nan")

# Split before fitting any preprocessing step.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Validate the split before modeling.
if X_train.shape[0] == 0 or X_test.shape[0] == 0:
    raise ValueError("Train and test sets must both contain rows.")

# Build base models that each preprocess only their training folds.
base_models = [
    ("tree", DecisionTreeClassifier(max_depth=4, random_state=42)),
    ("knn", make_pipeline(SimpleImputer(), StandardScaler(), KNeighborsClassifier())),
]

# Safe stacking keeps preprocessing inside cross-validation folds.
safe_stack = StackingClassifier(
    estimators=base_models,
    final_estimator=make_pipeline(
        SimpleImputer(),
        StandardScaler(),
        LogisticRegression(max_iter=500, random_state=42),
    ),
    passthrough=True,
    cv=5,
)

# Fit the safe stack only on training data.
safe_stack.fit(X_train, y_train)
safe_accuracy = accuracy_score(y_test, safe_stack.predict(X_test))

# This leaky version fits preprocessing before the train-test split.
leaky_preprocessor = make_pipeline(SimpleImputer(), StandardScaler())
leaky_features = leaky_preprocessor.fit_transform(features)

# The split reuses the same rows after leakage has already happened.
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    leaky_features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# The leaky stack receives passthrough features influenced by test rows.
leaky_stack = StackingClassifier(
    estimators=base_models,
    final_estimator=make_pipeline(
        SimpleImputer(),
        StandardScaler(),
        LogisticRegression(max_iter=500, random_state=42),
    ),
    passthrough=True,
    cv=5,
)

# Fit and compare the two workflows.
leaky_stack.fit(X_train_bad, y_train_bad)
leaky_accuracy = accuracy_score(y_test_bad, leaky_stack.predict(X_test_bad))

print("scikit-learn version:", sklearn.__version__)
print("Safe passthrough test accuracy:", round(safe_accuracy, 3))
print("Leaky passthrough test accuracy:", round(leaky_accuracy, 3))
print("Lesson: fit preprocessing inside the stacking workflow.")



### **3.3. Leakage Prevention**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_B/image_03_03.jpg?v=1784012388" width="250">



>* Train meta-models on out-of-fold predictions
>* Refit base models after leakage-safe training

>* Fit preprocessing within each training fold
>* Keep meta-features free from future information

>* Match validation splits to real prediction conditions
>* Nest preprocessing and keep final tests untouched



In [ ]:
#@title Python Code - Leakage Prevention

# This example contrasts leaky and safe stacking.
# Out-of-fold predictions protect the meta-model.
# The printed scores reveal optimistic leakage.

import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small classification dataset for stacking.
features, target = make_classification(
    n_samples=900,
    n_features=12,
    n_informative=6,
    random_state=42,
)

# Keep a final test set untouched until evaluation.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.30,
    stratify=target,
    random_state=42,
)

# Validate that the split kept matching row counts.
if len(train_features) != len(train_target):
    raise ValueError("Training features and labels must match.")

# Define one base model inside a preprocessing pipeline.
base_model = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(n_estimators=60, random_state=42),
)

# Leaky meta-features come from predictions on seen rows.
leaky_base = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(n_estimators=60, random_state=42),
)
leaky_base.fit(train_features, train_target)
leaky_train_meta = leaky_base.predict_proba(train_features)[:, 1].reshape(-1, 1)

# Safe meta-features come from out-of-fold predictions.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
safe_train_meta = cross_val_predict(
    base_model,
    train_features,
    train_target,
    cv=folds,
    method="predict_proba",
)[:, 1].reshape(-1, 1)

# Train one simple meta-model for each design.
leaky_meta_model = LogisticRegression(max_iter=300, random_state=42)
safe_meta_model = LogisticRegression(max_iter=300, random_state=42)
leaky_meta_model.fit(leaky_train_meta, train_target)
safe_meta_model.fit(safe_train_meta, train_target)

# Refit the base model only after meta-training is built.
final_base_model = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(n_estimators=60, random_state=42),
)
final_base_model.fit(train_features, train_target)
test_meta = final_base_model.predict_proba(test_features)[:, 1].reshape(-1, 1)

# Compare development scores with the untouched test score.
leaky_train_score = accuracy_score(
    train_target,
    leaky_meta_model.predict(leaky_train_meta),
)
safe_train_score = accuracy_score(train_target, safe_meta_model.predict(safe_train_meta))
safe_test_score = accuracy_score(test_target, safe_meta_model.predict(test_meta))

print("scikit-learn version:", sklearn.__version__)
print("Leaky stacking training accuracy:", round(leaky_train_score, 3))
print("Safe out-of-fold training accuracy:", round(safe_train_score, 3))
print("Safe stacking untouched test accuracy:", round(safe_test_score, 3))
print("Lesson: train meta-models on out-of-fold predictions only.")



# <font color="#418FDE" size="6.5" uppercase>**Meta Ensembles**</font>


In this lecture, you learned to:
- Apply AdaBoost and voting ensembles to combine base estimators. 
- Build stacked classifiers and regressors using cross-validated meta-features. 
- Prevent leakage when designing ensemble workflows. 

In the next Module (Module 19), we will go over 'Complex Outputs'